In [1]:
#  Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Set your project directory
import os

project_path = '/content/drive/MyDrive/Colab Notebooks'
os.chdir(project_path)

print(os.listdir())

['Neuron.ipynb', 'main', 'dataset', 'Text_Summarizer.ipynb']


In [3]:
!pip install transformers

In [4]:
!pip install "transformers[torch]"

In [5]:
# Import Libraries
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [29]:
# Load the data
train_data = pd.read_csv('dataset/samsum-train.csv')
val_data = pd.read_csv('dataset/samsum-validation.csv')


In [30]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [31]:
train_data['dialogue'][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [10]:
train_data.sample(5)

,id,dialogue,summary
3057,13717058,Kevin: Just tell me one thing - what are they ...,There was going to be a larynx cancer surgery ...
2081,13821827,Ron: Request: please lock the door to the whit...,Ron asks Ann and Sissy to lock the door to the...
8254,13809908,"Madison: Hey Will, what are you and Trina doin...",William and Madison are planning to meet up fo...
4880,13729594,Patric: Auuu!!\r\nJohn: Auu auuuuu!!!\r\nPatri...,Patric and John are meeting up at 9pm at Marke...
14246,13727715,Stella: I dread the thought of shopping tonigh...,Stella has to go shoppping tonight. She will o...


In [11]:
train_data.shape

(14732, 3)

In [12]:
val_data.shape

(818, 3)

In [32]:
# Random sampling for training data
train_data = train_data.sample(n=5000, random_state=42).reset_index(drop=True)

# Random sampling for validation data
valid_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [33]:
print(f"trained data size: {train_data.shape}, valid data size: {valid_data.shape}")

trained data size: (5000, 3), valid data size: (500, 3)


In [34]:
# Data Preprocessing
import re  # Regular Expression Library

def clean_data(text):
    text = re.sub(r"<.*?>", " ", text)  # Remove HTML tags
    text = re.sub(r"[\r\n]+", " ", text)  # Replace line breaks with spaces
    text = re.sub(r"\s+", " ", text)  # Remove extra spaces
    text = text.strip().lower()  # Remove surrounding spaces and convert to lowercase
    return text

In [35]:
# Clean Training Data
train_data['dialogue']=train_data['dialogue'].apply(clean_data)
train_data['summary']=train_data['summary'].apply(clean_data)

# Clean Validation Data
valid_data['dialogue']=valid_data['dialogue'].apply(clean_data)
valid_data['summary']=valid_data['summary'].apply(clean_data)


In [36]:
train_data['dialogue'][0]


"violet: hi! i came across this austin's article and i thought that you might find it interesting violet: claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

**Tokenization**

In [37]:
# from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration
tokenizer = T5Tokenizer.from_pretrained("t5-small") # light weight version for t5 model


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
# raw data to tokenized input for model fine tuning
#  take raw data and  tokenize both dialogue and summary and combine these for model training inputs
def tokenize(data):
  inputs=tokenizer(data['dialogue'],padding='max_length',max_length=512,truncation=True)
  targets=tokenizer(data['summary'],padding='max_length',max_length=150,truncation=True)
  # combine the target with inputs
  inputs['labels']=target['input_ids'] # token id -> add to  input as labels
  return inputs


In [ ]:
train_dataset=train_data.apply(tokenize, axis=1).tolist() # easy for huggingface
valid_dataset=valid_data.apply(tokenize, axis=1).tolist()

In [ ]:
train_dataset[0]

In [ ]:
valid_dataset[0]

In [ ]:
# inputs ids -> dialogue tokenized version
  # 1 is for End of Sequence and 0 are the padding

# attention mask -> which valid value in input and padding values

# labels - target -> summary token ids

In [ ]:
print(len(train_dataset[0]['input_ids']))
print(len(train_dataset[0]['labels']))


In [ ]:
type(train_dataset)
type(valid_dataset)